In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
# Montar o Drive para acessar/salvar os arquivos
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
CAMINHO_ENTRADA = "/content/drive/MyDrive/Iniciacao_cientifica/base de dados/dados 10.000 linhas/SAEB_ALUNO_9EF.csv"


In [ ]:
CAMINHO_SAIDA = '/content/drive/My Drive/Iniciacao_cientifica/base de dados/dados limpos/SAEB_ALUNO_9EF_tratado.csv'


In [ ]:
# Lista final de colunas mantidas (Baseado no foco do projeto: Aprendizagem x Socioeconômico x Saúde)
COLUNAS_MANTER_FINAL = [
    # IDENTIFICADORES E PESOS (Para Merge e Amostragem)
    'ID_ESCOLA', 'ID_ALUNO', 'ID_MUNICIPIO', 'ID_SERIE', 'ESTRATO',
    'PESO_ALUNO_LP', 'PESO_ALUNO_MT', 'PESO_ALUNO_INSE',

    # RESULTADO E CONTEXTO EDUCACIONAL
    'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_MT', 'ERRO_PADRAO_MT',
    'PROFICIENCIA_LP_SAEB', 'ERRO_PADRAO_LP_SAEB', 'PROFICIENCIA_MT_SAEB', 'ERRO_PADRAO_MT_SAEB',
    'INSE_ALUNO', 'NU_TIPO_NIVEL_INSE', 'IN_PUBLICA', 'ID_LOCALIZACAO',

    # VARIÁVEIS CHAVE DO QUESTIONÁRIO (Mantemos apenas até Q24, pois Q25 falhou)
    'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03', 'TX_RESP_Q04', 'TX_RESP_Q05a',
    'TX_RESP_Q05b', 'TX_RESP_Q05c', 'TX_RESP_Q06', 'TX_RESP_Q07a', 'TX_RESP_Q07b',
    'TX_RESP_Q07c', 'TX_RESP_Q07d', 'TX_RESP_Q07e', 'TX_RESP_Q08', 'TX_RESP_Q09',
    'TX_RESP_Q10a', 'TX_RESP_Q10b', 'TX_RESP_Q10c', 'TX_RESP_Q10d', 'TX_RESP_Q10e',
    'TX_RESP_Q10f', 'TX_RESP_Q11a', 'TX_RESP_Q11b', 'TX_RESP_Q11c', 'TX_RESP_Q12a',
    'TX_RESP_Q12b', 'TX_RESP_Q12c', 'TX_RESP_Q12d', 'TX_RESP_Q12e', 'TX_RESP_Q12f',
    'TX_RESP_Q12g', 'TX_RESP_Q13a', 'TX_RESP_Q13b', 'TX_RESP_Q13c', 'TX_RESP_Q13d',
    'TX_RESP_Q13e', 'TX_RESP_Q13f', 'TX_RESP_Q13g', 'TX_RESP_Q13h', 'TX_RESP_Q13i',
    'TX_RESP_Q14', 'TX_RESP_Q15a', 'TX_RESP_Q15b', 'TX_RESP_Q16', 'TX_RESP_Q17',
    'TX_RESP_Q18', 'TX_RESP_Q19', 'TX_RESP_Q20', 'TX_RESP_Q21a', 'TX_RESP_Q21b',
    'TX_RESP_Q21c', 'TX_RESP_Q21d', 'TX_RESP_Q21e', 'TX_RESP_Q22a', 'TX_RESP_Q22b',
    'TX_RESP_Q22c', 'TX_RESP_Q22d', 'TX_RESP_Q22e', 'TX_RESP_Q22f', 'TX_RESP_Q22g',
    'TX_RESP_Q22h', 'TX_RESP_Q23a', 'TX_RESP_Q23b', 'TX_RESP_Q23c', 'TX_RESP_Q23d',
    'TX_RESP_Q23e', 'TX_RESP_Q23f', 'TX_RESP_Q23g', 'TX_RESP_Q23h', 'TX_RESP_Q23i',
    'TX_RESP_Q24'
]

COLUNAS_NUMERICAS = [
    'PESO_ALUNO_LP', 'PESO_ALUNO_MT', 'PESO_ALUNO_INSE',
    'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_MT', 'ERRO_PADRAO_MT',
    'PROFICIENCIA_LP_SAEB', 'ERRO_PADRAO_LP_SAEB', 'PROFICIENCIA_MT_SAEB', 'ERRO_PADRAO_MT_SAEB',
    'INSE_ALUNO'
]

VALORES_AUSENTES = ['.', '..', '...', '....', ' ', '']

**#PROCESSO DE LIMPEZA**

In [ ]:
print(f"Iniciando tratamento do {CAMINHO_ENTRADA}...")

Iniciando tratamento do /content/drive/MyDrive/Iniciacao_cientifica/base de dados/dados 10.000 linhas/SAEB_ALUNO_9EF.csv...


In [ ]:
try:
    # 2. Leitura do arquivo bruto
    df = pd.read_csv(CAMINHO_ENTRADA, sep=',', encoding='latin-1', na_values=VALORES_AUSENTES)

    # 3. Seleção e Filtro das Colunas
    # Utilziar o método 'reindex' com 'axis=1' para selecionar colunas. Se uma coluna não existir, ela será IGNORADA.
    df_tratado = df[COLUNAS_MANTER_FINAL]

    # 4. Conversão de Tipos e Tratamento de Missing
    for col in COLUNAS_NUMERICAS:
        df_tratado[col] = pd.to_numeric(df_tratado[col], errors='coerce')

    # 5. Tratamento Categórico (Garantir IDs como inteiros)
    df_tratado['ID_ESCOLA'] = df_tratado['ID_ESCOLA'].astype('Int64', errors='ignore')
    df_tratado['ID_ALUNO'] = df_tratado['ID_ALUNO'].astype('Int64', errors='ignore')
    df_tratado['ID_MUNICIPIO'] = df_tratado['ID_MUNICIPIO'].astype('Int64', errors='ignore')

    # 6. Remoção de Duplicatas
    df_tratado = df_tratado.drop_duplicates(subset=['ID_ALUNO'], keep='first')
    print(f"Base final com {df_tratado.shape[0]} registros e {df_tratado.shape[1]} colunas.")

    # 7. Salvar o arquivo tratado e filtrado
    df_tratado.to_csv(CAMINHO_SAIDA, index=False)
    print(f"\n SUCESSO: Arquivo tratado e filtrado salvo em: {CAMINHO_SAIDA}")

except KeyError as e:
    print(f" ERRO: A coluna {e} ainda está faltando. Vamos remover ela da lista para seguir adiante.")
except Exception as e:
    print(f"\n ERRO FATAL: {e}")

/tmp/ipython-input-1396139209.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tratado[col] = pd.to_numeric(df_tratado[col], errors='coerce')
/tmp/ipython-input-1396139209.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tratado['ID_ESCOLA'] = df_tratado['ID_ESCOLA'].astype('Int64', errors='ignore')
/tmp/ipython-input-1396139209.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

Base final com 10000 registros e 91 colunas.

✅ SUCESSO: Arquivo tratado e filtrado salvo em: /content/drive/My Drive/Iniciacao_cientifica/base de dados/dados limpos/SAEB_ALUNO_9EF_tratado.csv
